# 06 — V3 Class-Conditional Latent Alignment

## Purpose
V2 maps heterogeneous client inputs into the same 8-D latent size, but equal dimensionality does not guarantee semantic alignment. V3 tests one targeted extension:

**V3 = V2 + class-conditional latent prototype alignment**

Each client computes an aggregate latent prototype for class 0 and class 1. During federated training, each private encoder is encouraged to move its same-class latent representation toward the other client's aggregate prototype. Encoders remain private; only compact prototypes are exchanged conceptually, and the compatible shared predictor is still aggregated.

### Predefined question
Does alignment reduce the **Federated − Local** loss on Dataset 1 without materially harming Dataset 2?

This is the final major modelling experiment; results are retained whether or not V3 improves performance.


In [1]:
%pip install scikit-learn torch

import os, copy, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SPLIT_SEED = 42
SEEDS = [42, 52, 62, 72, 82]
LATENT_DIM = 8
ALIGN_LAMBDA = 0.10  # predefined; do not tune on the test set
print("Seeds:", SEEDS, "| alignment lambda:", ALIGN_LAMBDA)


Note: you may need to restart the kernel to use updated packages.
Seeds: [42, 52, 62, 72, 82] | alignment lambda: 0.1


## 1. Load data and preserve the V2 split protocol


In [2]:
DATASET1_PATH = "../data/diabetes_prediction_dataset.csv"
DATASET2_PATH = "../data/diabetes_dataset.csv"
df1 = pd.read_csv(DATASET1_PATH)
df2 = pd.read_csv(DATASET2_PATH)
TARGET1, TARGET2 = "diabetes", "diagnosed_diabetes"

df2_model = df2.drop(columns=[c for c in ["diabetes_stage","diabetes_risk_score"] if c in df2.columns]).copy()
X1, y1 = df1.drop(columns=[TARGET1]), df1[TARGET1].astype(int)
X2, y2 = df2_model.drop(columns=[TARGET2]), df2_model[TARGET2].astype(int)

X1_full, X1_test, y1_full, y1_test = train_test_split(X1,y1,test_size=.20,random_state=SPLIT_SEED,stratify=y1)
X2_full, X2_test, y2_full, y2_test = train_test_split(X2,y2,test_size=.20,random_state=SPLIT_SEED,stratify=y2)
X1_train,X1_val,y1_train,y1_val = train_test_split(X1_full,y1_full,test_size=.125,random_state=SPLIT_SEED,stratify=y1_full)
X2_train,X2_val,y2_train,y2_val = train_test_split(X2_full,y2_full,test_size=.125,random_state=SPLIT_SEED,stratify=y2_full)
print("D1:",len(X1_train),len(X1_val),len(X1_test))
print("D2:",len(X2_train),len(X2_val),len(X2_test))


D1: 70000 10000 20000
D2: 70000 10000 20000


## 2. Client-specific preprocessing


In [3]:
def build_preprocessor(X):
    cat=X.select_dtypes(include=["object","category","bool"]).columns.tolist()
    num=[c for c in X.columns if c not in cat]
    return ColumnTransformer([
        ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),num),
        ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),cat)
    ])

prep1,prep2=build_preprocessor(X1_train),build_preprocessor(X2_train)
A_train=prep1.fit_transform(X1_train).astype(np.float32); A_val=prep1.transform(X1_val).astype(np.float32); A_test=prep1.transform(X1_test).astype(np.float32)
B_train=prep2.fit_transform(X2_train).astype(np.float32); B_val=prep2.transform(X2_val).astype(np.float32); B_test=prep2.transform(X2_test).astype(np.float32)
yA_train=np.asarray(y1_train,dtype=np.int64); yB_train=np.asarray(y2_train,dtype=np.int64)
print("Input dims:",A_train.shape[1],B_train.shape[1])


Input dims: 15 46


## 3. Same V2 architecture and utilities


In [4]:
class ClientEncoder(nn.Module):
    def __init__(self,input_dim,latent_dim=LATENT_DIM):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(input_dim,32),nn.ReLU(),nn.Linear(32,latent_dim),nn.ReLU())
    def forward(self,x): return self.net(x)

class SharedPredictor(nn.Module):
    def __init__(self,latent_dim=LATENT_DIM):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(latent_dim,8),nn.ReLU(),nn.Linear(8,1))
    def forward(self,z): return self.net(z).squeeze(1)

class ClientModel(nn.Module):
    def __init__(self,encoder,predictor):
        super().__init__(); self.encoder=encoder; self.predictor=predictor
    def forward(self,x): return self.predictor(self.encoder(x))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def loader(X,y,batch=512):
    return DataLoader(TensorDataset(torch.tensor(X,dtype=torch.float32),torch.tensor(np.asarray(y),dtype=torch.float32)),batch_size=batch,shuffle=True)

def train_standard(model,X,y,epochs=5,lr=1e-3):
    opt=torch.optim.Adam(model.parameters(),lr=lr); loss_fn=nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb,yb in loader(X,y):
            opt.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward(); opt.step()
    return model

def predict_prob(model,X):
    model.eval()
    with torch.no_grad(): return torch.sigmoid(model(torch.tensor(X,dtype=torch.float32))).cpu().numpy()

def best_threshold(y,p):
    ts=np.linspace(.05,.95,181)
    fs=[f1_score(y,(p>=t).astype(int),zero_division=0) for t in ts]
    return float(ts[int(np.argmax(fs))])

def get_metrics(y,p,t):
    pred=(p>=t).astype(int)
    return {"Threshold":t,"Accuracy":accuracy_score(y,pred),"Precision":precision_score(y,pred,zero_division=0),
            "Recall":recall_score(y,pred,zero_division=0),"F1":f1_score(y,pred,zero_division=0),"ROC-AUC":roc_auc_score(y,p)}

def average_predictors(ps,ns):
    total=float(sum(ns)); state=copy.deepcopy(ps[0].state_dict())
    for k in state:
        state[k]=torch.zeros_like(state[k])
        for p,n in zip(ps,ns): state[k]+=p.state_dict()[k]*(n/total)
    result=SharedPredictor(); result.load_state_dict(state); return result


## 4. Class-conditional latent alignment
A prototype is the mean encoder output for one class. The other client's prototype is detached, so each client updates only its own encoder toward an aggregate target; there is no backpropagation through the other client.


In [5]:
def prototypes(encoder,X,y):
    encoder.eval()
    with torch.no_grad():
        z=encoder(torch.tensor(X,dtype=torch.float32))
        yt=torch.tensor(np.asarray(y),dtype=torch.long)
        return {c:z[yt==c].mean(0).detach().clone() for c in [0,1]}

def train_aligned(model,X,y,target,epochs=3,lr=1e-3,align_lambda=ALIGN_LAMBDA):
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    bce,mse=nn.BCEWithLogitsLoss(),nn.MSELoss()
    model.train()
    for _ in range(epochs):
        for xb,yb in loader(X,y):
            opt.zero_grad()
            z=model.encoder(xb); logits=model.predictor(z)
            pred_loss=bce(logits,yb)
            terms=[]
            yl=yb.long()
            for c in [0,1]:
                mask=yl==c
                if mask.sum()>0: terms.append(mse(z[mask].mean(0),target[c]))
            align_loss=torch.stack(terms).mean()
            (pred_loss+align_lambda*align_loss).backward(); opt.step()
    return model


## 5. One seed: unchanged local control vs aligned federation


In [6]:
def run_seed(seed,rounds=5,local_epochs=3):
    set_seed(seed)
    localA=ClientModel(ClientEncoder(A_train.shape[1]),SharedPredictor())
    localB=ClientModel(ClientEncoder(B_train.shape[1]),SharedPredictor())
    train_standard(localA,A_train,y1_train,epochs=20)
    train_standard(localB,B_train,y2_train,epochs=20)

    set_seed(seed)
    encA,encB=ClientEncoder(A_train.shape[1]),ClientEncoder(B_train.shape[1])
    global_pred=SharedPredictor()

    for _ in range(rounds):
        protoA,protoB=prototypes(encA,A_train,yA_train),prototypes(encB,B_train,yB_train)
        modelA=ClientModel(encA,copy.deepcopy(global_pred))
        modelB=ClientModel(encB,copy.deepcopy(global_pred))
        train_aligned(modelA,A_train,yA_train,protoB,epochs=local_epochs)
        train_aligned(modelB,B_train,yB_train,protoA,epochs=local_epochs)
        encA,encB=modelA.encoder,modelB.encoder
        global_pred=average_predictors([modelA.predictor,modelB.predictor],[len(A_train),len(B_train)])

    fedA=ClientModel(encA,copy.deepcopy(global_pred)); fedB=ClientModel(encB,copy.deepcopy(global_pred))
    cases=[
        ("Local","Dataset 1",localA,A_val,y1_val,A_test,y1_test),
        ("Local","Dataset 2",localB,B_val,y2_val,B_test,y2_test),
        ("Federated V3","Dataset 1",fedA,A_val,y1_val,A_test,y1_test),
        ("Federated V3","Dataset 2",fedB,B_val,y2_val,B_test,y2_test)]
    rows=[]
    for approach,dataset,model,Xv,yv,Xt,yt in cases:
        pv=predict_prob(model,Xv); t=best_threshold(yv,pv); pt=predict_prob(model,Xt)
        rows.append({"Seed":seed,"Approach":approach,"Dataset":dataset,**get_metrics(yt,pt,t)})
    return rows


## 6. Run the same five seeds


In [7]:
rows=[]
for i,s in enumerate(SEEDS,1):
    print(f"Running seed {s} ({i}/{len(SEEDS)})...")
    rows.extend(run_seed(s))
    print("Completed",s)
v3_results=pd.DataFrame(rows)
metric_cols=["Accuracy","Precision","Recall","F1","ROC-AUC"]
display(v3_results.round(4))


Running seed 42 (1/5)...
Completed 42
Running seed 52 (2/5)...
Completed 52
Running seed 62 (3/5)...
Completed 62
Running seed 72 (4/5)...
Completed 72
Running seed 82 (5/5)...
Completed 82


,Seed,Approach,Dataset,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC
0,42,Local,Dataset 1,0.415,0.9698,0.9329,0.6953,0.7968,0.9758
1,42,Local,Dataset 2,0.355,0.9096,0.9784,0.8684,0.9201,0.9416
2,42,Federated V3,Dataset 1,0.630,0.9710,0.9651,0.6835,0.8003,0.9741
3,42,Federated V3,Dataset 2,0.240,0.9156,0.9952,0.8635,0.9247,0.9420
4,52,Local,Dataset 1,0.470,0.9684,0.9428,0.6694,0.7829,0.9747
5,52,Local,Dataset 2,0.455,0.9098,0.9879,0.8601,0.9196,0.9420
6,52,Federated V3,Dataset 1,0.785,0.9708,0.9673,0.6794,0.7982,0.9743
7,52,Federated V3,Dataset 2,0.225,0.9136,0.9910,0.8639,0.9231,0.9418
8,62,Local,Dataset 1,0.460,0.9700,0.9568,0.6776,0.7934,0.9732
9,62,Local,Dataset 2,0.570,0.9137,0.9936,0.8618,0.9230,0.9419


## 7. Mean ± standard deviation


In [8]:
summary=[]
for (dataset,approach),g in v3_results.groupby(["Dataset","Approach"]):
    r={"Dataset":dataset,"Approach":approach}
    for m in metric_cols: r[m]=f"{g[m].mean():.4f} ± {g[m].std(ddof=1):.4f}"
    summary.append(r)
v3_summary=pd.DataFrame(summary)
display(v3_summary)


,Dataset,Approach,Accuracy,Precision,Recall,F1,ROC-AUC
0,Dataset 1,Federated V3,0.9710 ± 0.0006,0.9671 ± 0.0167,0.6819 ± 0.0059,0.7997 ± 0.0020,0.9746 ± 0.0011
1,Dataset 1,Local,0.9688 ± 0.0020,0.9379 ± 0.0212,0.6775 ± 0.0122,0.7867 ± 0.0131,0.9736 ± 0.0018
2,Dataset 2,Federated V3,0.9132 ± 0.0015,0.9904 ± 0.0032,0.8638 ± 0.0011,0.9227 ± 0.0012,0.9420 ± 0.0004
3,Dataset 2,Local,0.9112 ± 0.0017,0.9883 ± 0.0060,0.8623 ± 0.0035,0.9210 ± 0.0013,0.9417 ± 0.0003


## 8. Federated V3 − Local deltas


In [9]:
p=v3_results.pivot(index=["Seed","Dataset"],columns="Approach",values=metric_cols)
d=[]
for s in SEEDS:
    for ds in ["Dataset 1","Dataset 2"]:
        r={"Seed":s,"Dataset":ds}
        for m in metric_cols: r["Delta "+m]=p.loc[(s,ds),(m,"Federated V3")]-p.loc[(s,ds),(m,"Local")]
        d.append(r)
v3_deltas=pd.DataFrame(d)
display(v3_deltas.round(4))
display(v3_deltas.groupby("Dataset")[[c for c in v3_deltas if c.startswith("Delta")]].agg(["mean","std"]).round(4))


,Seed,Dataset,Delta Accuracy,Delta Precision,Delta Recall,Delta F1,Delta ROC-AUC
0,42,Dataset 1,0.0011,0.0322,-0.0118,0.0035,-0.0017
1,42,Dataset 2,0.0060,0.0168,-0.0049,0.0045,0.0003
2,52,Dataset 1,0.0023,0.0245,0.0100,0.0153,-0.0004
3,52,Dataset 2,0.0039,0.0031,0.0038,0.0035,-0.0002
4,62,Dataset 1,0.0002,-0.0116,0.0118,0.0039,0.0024
5,62,Dataset 2,-0.0015,-0.0070,0.0036,-0.0010,-0.0005
6,72,Dataset 1,0.0064,0.0884,0.0100,0.0371,0.0022
7,72,Dataset 2,0.0010,-0.0006,0.0022,0.0010,0.0010
8,82,Dataset 1,0.0010,0.0128,0.0018,0.0056,0.0026
9,82,Dataset 2,0.0004,-0.0019,0.0024,0.0006,0.0007


Delta Accuracy         Delta Precision        Delta Recall          \
                    mean     std            mean    std         mean     std   
Dataset                                                                        
Dataset 1         0.0022  0.0025          0.0292  0.037       0.0044  0.0098   
Dataset 2         0.0020  0.0030          0.0021  0.009       0.0014  0.0036   

          Delta F1         Delta ROC-AUC          
              mean     std          mean     std  
Dataset                                           
Dataset 1   0.0131  0.0143        0.0010  0.0020  
Dataset 2   0.0017  0.0023        0.0003  0.0006

## 9. Direct V3 vs V2 comparison


In [10]:
V2_PATH="../models/prototype_v2_multiseed_results.csv"
if os.path.exists(V2_PATH):
    v2=pd.read_csv(V2_PATH)
    v2f=v2[v2.Approach=="Federated"][["Seed","Dataset"]+metric_cols].rename(columns={m:"V2_Fed_"+m for m in metric_cols})
    v3f=v3_results[v3_results.Approach=="Federated V3"][["Seed","Dataset"]+metric_cols].rename(columns={m:"V3_Fed_"+m for m in metric_cols})
    v2_v3=v2f.merge(v3f,on=["Seed","Dataset"])
    for m in metric_cols: v2_v3["V3_minus_V2_"+m]=v2_v3["V3_Fed_"+m]-v2_v3["V2_Fed_"+m]
    display(v2_v3.round(4))
    cols=["V3_minus_V2_"+m for m in metric_cols]
    display(v2_v3.groupby("Dataset")[cols].agg(["mean","std"]).round(4))
else:
    v2_v3=None
    print("V2 CSV not found:",V2_PATH)


,Seed,Dataset,V2_Fed_Accuracy,V2_Fed_Precision,V2_Fed_Recall,V2_Fed_F1,V2_Fed_ROC-AUC,V3_Fed_Accuracy,V3_Fed_Precision,V3_Fed_Recall,V3_Fed_F1,V3_Fed_ROC-AUC,V3_minus_V2_Accuracy,V3_minus_V2_Precision,V3_minus_V2_Recall,V3_minus_V2_F1,V3_minus_V2_ROC-AUC
0,42,Dataset 1,0.9678,0.9483,0.6576,0.7767,0.9636,0.9710,0.9651,0.6835,0.8003,0.9741,0.0031,0.0169,0.0259,0.0236,0.0105
1,42,Dataset 2,0.9114,0.9885,0.8622,0.9211,0.9421,0.9156,0.9952,0.8635,0.9247,0.9420,0.0042,0.0067,0.0013,0.0036,-0.0001
2,52,Dataset 1,0.9703,0.9784,0.6653,0.7920,0.9709,0.9708,0.9673,0.6794,0.7982,0.9743,0.0005,-0.0110,0.0141,0.0062,0.0033
3,52,Dataset 2,0.9113,0.9895,0.8613,0.9210,0.9415,0.9136,0.9910,0.8639,0.9231,0.9418,0.0023,0.0015,0.0026,0.0021,0.0003
4,62,Dataset 1,0.9620,0.8666,0.6535,0.7451,0.9673,0.9702,0.9452,0.6894,0.7973,0.9756,0.0082,0.0785,0.0359,0.0521,0.0083
5,62,Dataset 2,0.9122,0.9849,0.8669,0.9221,0.9419,0.9122,0.9866,0.8653,0.9220,0.9414,0.0000,0.0018,-0.0016,-0.0001,-0.0005
6,72,Dataset 1,0.9703,0.9431,0.6924,0.7985,0.9713,0.9718,0.9922,0.6735,0.8024,0.9732,0.0015,0.0491,-0.0188,0.0039,0.0020
7,72,Dataset 2,0.9122,0.9900,0.8622,0.9217,0.9415,0.9123,0.9886,0.8638,0.9220,0.9425,0.0001,-0.0014,0.0015,0.0002,0.0010
8,82,Dataset 1,0.9712,0.9707,0.6812,0.8006,0.9724,0.9710,0.9659,0.6835,0.8006,0.9760,-0.0001,-0.0047,0.0024,-0.0000,0.0035
9,82,Dataset 2,0.9130,0.9930,0.8610,0.9223,0.9417,0.9124,0.9903,0.8623,0.9219,0.9421,-0.0006,-0.0027,0.0013,-0.0004,0.0004


V3_minus_V2_Accuracy         V3_minus_V2_Precision          \
                          mean     std                  mean     std   
Dataset                                                                
Dataset 1               0.0026  0.0033                0.0257  0.0377   
Dataset 2               0.0012  0.0020                0.0012  0.0036   

          V3_minus_V2_Recall         V3_minus_V2_F1          \
                        mean     std           mean     std   
Dataset                                                       
Dataset 1             0.0119  0.0213         0.0172  0.0215   
Dataset 2             0.0010  0.0015         0.0011  0.0017   

          V3_minus_V2_ROC-AUC          
                         mean     std  
Dataset                                
Dataset 1              0.0055  0.0037  
Dataset 2              0.0002  0.0006

## 10. Save V3 outputs


In [11]:
os.makedirs("../models",exist_ok=True)
v3_results.to_csv("../models/prototype_v3_multiseed_results.csv",index=False)
v3_summary.to_csv("../models/prototype_v3_multiseed_summary.csv",index=False)
v3_deltas.to_csv("../models/prototype_v3_multiseed_deltas.csv",index=False)
if v2_v3 is not None: v2_v3.to_csv("../models/prototype_v3_vs_v2.csv",index=False)
print("Saved V3 outputs.")


Saved V3 outputs.
